In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np

from sdp.data import taxonomy as tax
from sdp.data.labeling import tier1
from sdp.data import splitting as sp
from sdp.data import sampling as smp

ROOT    = Path.cwd().parent
RAW     = ROOT / "data" / "raw" / "codenet_cpp_metadata.parquet"
INTERIM = ROOT / "data" / "interim"
REPORTS = ROOT / "reports"
FIG     = ROOT / "reports" / "figures"

In [2]:
status = pd.read_parquet(RAW, columns=["status"])["status"]
print(f"Rows: {len(status):,}")

# Raises UnknownVerdictError, naming every unrecognised verdict at once.
tier1.validate_vocabulary(status.unique())
print("Vocabulary validated: all verdicts documented.")

counts = status.value_counts()
discards = pd.DataFrame({
    "verdict": [v for v in counts.index if v in tier1.DISCARDED_VERDICTS],
})
discards["reason"] = discards["verdict"].map(tier1.VERDICT_TO_DISCARD_REASON).astype(str)
discards["count"]  = discards["verdict"].map(counts)
discards["pct_of_cpp"] = (discards["count"] / len(status) * 100).round(4)
discards = discards.sort_values("count", ascending=False).reset_index(drop=True)

n_kept = int(counts[list(tier1.KEPT_VERDICTS)].sum())
print(discards.to_string(index=False))
print(f"\nKept:      {n_kept:,}  ({n_kept/len(status)*100:.2f}%)")
print(f"Discarded: {len(status)-n_kept:,}  ({(len(status)-n_kept)/len(status)*100:.2f}%)")

del status
gc.collect()

Rows: 8,008,527
Vocabulary validated: all verdicts documented.
               verdict                reason  count  pct_of_cpp
   Time Limit Exceeded EFFICIENCY_CONSTRAINT 326340      4.0749
WA: Presentation Error         OUTPUT_FORMAT  26449      0.3303
 Memory Limit Exceeded EFFICIENCY_CONSTRAINT  14637      0.1828
 Output Limit Exceeded EFFICIENCY_CONSTRAINT    778      0.0097
   Judge Not Available  JUDGE_INFRASTRUCTURE     94      0.0012
  Query Limit Exceeded EFFICIENCY_CONSTRAINT     88      0.0011
        Internal error  JUDGE_INFRASTRUCTURE     78       0.001
    Judge System Error  JUDGE_INFRASTRUCTURE      7      0.0001

Kept:      7,640,056  (95.40%)
Discarded: 368,471  (4.60%)


0

In [3]:
ext = pd.read_parquet(RAW, columns=["filename_ext", "language"])
print("filename_ext values:")
print(ext["filename_ext"].value_counts().to_string())
print("\nlanguage values:")
print(ext["language"].value_counts().to_string())

del ext
gc.collect()

filename_ext values:
filename_ext
cpp    8008527

language values:
language
C++    8008527


0

In [4]:
COLUMNS = [
    "submission_id",       # primary key, part of the file path
    "problem_id",          # the split key
    "user_id",             # identifies resubmission chains
    "status",              # provenance for the label
    "date",                # temporal reporting
    "code_size",           # truncation analysis; cross-check on extraction
    "original_language",   # Tier 2 depends on this (GCC 5.4.1 vs 9.2.1 vs Clang)
]

df = pd.read_parquet(
    RAW,
    columns=COLUMNS,
    filters=[("status", "in", sorted(tier1.KEPT_VERDICTS))],
)
print(f"Rows read: {len(df):,}")

df["coarse_label"] = pd.Categorical(
    df["status"].map(tier1.VERDICT_TO_CLASS).astype(str),
    categories=[c.value for c in tax.COARSE_ORDER],
    ordered=True,
)

for col in ["problem_id", "user_id", "status", "original_language"]:
    df[col] = df[col].astype("category")
df["submission_id"] = df["submission_id"].astype("string[pyarrow]")
df["code_size"] = df["code_size"].astype("int32")

print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
df.head(3)

Rows read: 7,640,056
Memory: 0.32 GB


,submission_id,problem_id,user_id,status,date,code_size,original_language,coarse_label
0,s317469200,p00000,u972675635,Compile Error,1530881659,250,C++,COMPILE_ERROR
1,s667847559,p00000,u972675635,Accepted,1530881789,250,C++11,ERROR_FREE
2,s160425098,p00000,u642752018,Accepted,1530897493,232,C++,ERROR_FREE


In [5]:
EXPECTED = {
    "ERROR_FREE":    4_353_049,
    "LOGICAL":       2_571_284,
    "COMPILE_ERROR":   376_053,
    "RUNTIME_ERROR":   339_670,
}

assert len(df) == 7_640_056, f"row count mismatch: {len(df):,}"
assert df["coarse_label"].isna().sum() == 0, "unlabelled rows present"
assert not df["submission_id"].duplicated().any(), "duplicate submission_id"

actual = df["coarse_label"].value_counts().sort_index()
print(actual.to_string())
assert {k: int(v) for k, v in actual.items()} == EXPECTED, "label distribution mismatch"

print(f"\nProblems: {df['problem_id'].nunique():,}")
print(f"Users:    {df['user_id'].nunique():,}")
print("\nAll checks passed.")

coarse_label
ERROR_FREE       4353049
COMPILE_ERROR     376053
RUNTIME_ERROR     339670
LOGICAL          2571284

Problems: 4,032
Users:    102,527

All checks passed.


In [6]:
MANIFEST = INTERIM / "labeled_manifest.parquet"

df.to_parquet(MANIFEST, compression="zstd", index=False)
discards.to_csv(REPORTS / "discard_summary.csv", index=False)

print(f"{MANIFEST.name}: {MANIFEST.stat().st_size/1e6:.1f} MB")
print(f"discard_summary.csv written ({len(discards)} rows)")

labeled_manifest.parquet: 92.0 MB
discard_summary.csv written (8 rows)


In [7]:
from sdp.data import splitting as sp

df = pd.read_parquet(INTERIM / "labeled_manifest.parquet")
print(f"Rows: {len(df):,}   problem_id dtype: {df['problem_id'].dtype}")

df["split"]        = sp.problem_level_split(df["problem_id"], seed=sp.DEFAULT_SEED)
df["random_split"] = sp.submission_level_split(df.index,      seed=sp.DEFAULT_SEED)

print(f"Seed: {sp.DEFAULT_SEED}   ratios: "
      f"{ {k.value: v for k, v in sp.DEFAULT_RATIOS.items()} }")
df[["submission_id", "problem_id", "coarse_label", "split", "random_split"]].head(3)

Rows: 7,640,056   problem_id dtype: category
Seed: 42   ratios: {'train': 0.6, 'val': 0.2, 'test': 0.2}


,submission_id,problem_id,coarse_label,split,random_split
0,s317469200,p00000,COMPILE_ERROR,val,val
1,s667847559,p00000,ERROR_FREE,val,train
2,s160425098,p00000,ERROR_FREE,val,train


In [10]:
import importlib
from sdp.data import splitting as sp
importlib.reload(sp)

<module 'sdp.data.splitting' from 'D:\\Dev\\Github\\transformer-defect-prediction\\src\\sdp\\data\\splitting.py'>

In [11]:
sp.assert_problem_disjoint(df["problem_id"], df["split"])
print("Disjointness: PASSED\n")

print("Achieved proportions and class counts")
print(sp.split_summary(df["split"], df["coarse_label"]).to_string())

print("\nProblems per split")
print(df.groupby("split", observed=True)["problem_id"].nunique().to_string())

print("\nClass composition within each split (% of split)")
ct = pd.crosstab(df["split"], df["coarse_label"], normalize="index") * 100
print(ct.round(2).to_string())

print("\nCorpus-wide reference (%)")
print((df["coarse_label"].value_counts(normalize=True).sort_index() * 100)
      .round(2).to_string())

Disjointness: PASSED

Achieved proportions and class counts
          rows   pct  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                                  
train  4584034  60.0     2571993         228192         199961  1583888
val    1528011  20.0      896626          76498          67385   487502
test   1528011  20.0      884430          71363          72324   499894

Problems per split
split
train    1528
val      1252
test     1252

Class composition within each split (% of split)
coarse_label  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                          
train              56.11           4.98           4.36    34.55
val                58.68           5.01           4.41    31.90
test               57.88           4.67           4.73    32.72

Corpus-wide reference (%)
coarse_label
ERROR_FREE       56.98
COMPILE_ERROR     4.92
RUNTIME_ERROR     4.45
LOGICAL          33.66


In [12]:
SPLITS = ROOT / "data" / "processed" / "splits"
SPLITS.mkdir(parents=True, exist_ok=True)

SPLIT_MANIFEST = SPLITS / "split_manifest.parquet"
df.to_parquet(SPLIT_MANIFEST, compression="zstd", index=False)

print(f"{SPLIT_MANIFEST.name}: {SPLIT_MANIFEST.stat().st_size/1e6:.1f} MB")
print(f"Columns: {list(df.columns)}")

split_manifest.parquet: 93.5 MB
Columns: ['submission_id', 'problem_id', 'user_id', 'status', 'date', 'code_size', 'original_language', 'coarse_label', 'split', 'random_split']


In [4]:
from sdp.data import sampling as smp

df = pd.read_parquet(ROOT / "data" / "processed" / "splits" / "split_manifest.parquet")
print(f"Manifest: {len(df):,} rows")

quotas = smp.resolve_quotas()
sample, report = smp.draw_sample(df, quotas=quotas, seed=smp.DEFAULT_SEED)

print(f"Sampled: {len(sample):,} rows   (seed {smp.DEFAULT_SEED})\n")
print(report.to_string(index=False))

Manifest: 7,640,056 rows
Sampled: 75,000 rows   (seed 42)

split  coarse_label  requested  available  taken  shortfall  problems  max_per_problem
train    ERROR_FREE       6000    2571993   6000          0      1523                4
train COMPILE_ERROR      18000     228192  18000          0      1314               18
train RUNTIME_ERROR      15000     199961  15000          0      1260               16
train       LOGICAL       6000    1583888   6000          0      1412                5
  val    ERROR_FREE       2000     896626   2000          0      1248                2
  val COMPILE_ERROR       6000      76498   6000          0      1045                7
  val RUNTIME_ERROR       5000      67385   5000          0      1006                6
  val       LOGICAL       2000     487502   2000          0      1137                2
 test    ERROR_FREE       2000     884430   2000          0      1246                2
 test COMPILE_ERROR       6000      71363   6000          0      1048  

In [5]:
assert report["shortfall"].sum() == 0, "supply shortfall — investigate before extracting"
assert len(sample) == 75_000, f"expected 75,000; got {len(sample):,}"
assert not sample["submission_id"].duplicated().any()

n_problems = sample["problem_id"].astype(str).nunique()
print(f"Distinct problems in the working corpus: {n_problems:,} of 4,032 "
      f"({n_problems/4032*100:.1f}%)")

print("\nRows per problem across the whole corpus")
per_problem = sample["problem_id"].astype(str).value_counts()
print(per_problem.describe(percentiles=[.5, .9, .99]).round(1).to_string())

print("\nCross-check: split x class counts")
print(pd.crosstab(sample["split"], sample["coarse_label"]).to_string())

sp.assert_problem_disjoint(sample["problem_id"], sample["split"])
print("\nDisjointness preserved in the sample: PASSED")

Distinct problems in the working corpus: 4,032 of 4,032 (100.0%)

Rows per problem across the whole corpus
count    4032.0
mean       18.6
std        13.0
min         1.0
50%        16.0
90%        42.0
99%        43.0
max        43.0

Cross-check: split x class counts
coarse_label  ERROR_FREE  COMPILE_ERROR  RUNTIME_ERROR  LOGICAL
split                                                          
train               6000          18000          15000     6000
val                 2000           6000           5000     2000
test                2000           6000           5000     2000

Disjointness preserved in the sample: PASSED


In [6]:
sample = sample.copy()
sample["archive_path"] = smp.archive_paths(sample)

SAMPLE_MANIFEST = INTERIM / "sample_manifest.parquet"
WANTED_PATHS    = INTERIM / "wanted_paths.txt"

sample.to_parquet(SAMPLE_MANIFEST, compression="zstd", index=False)
WANTED_PATHS.write_text("\n".join(sample["archive_path"]) + "\n", encoding="utf-8")

report.to_csv(REPORTS / "sampling_report.csv", index=False)

print(f"{SAMPLE_MANIFEST.name}: {SAMPLE_MANIFEST.stat().st_size/1e6:.2f} MB")
print(f"{WANTED_PATHS.name}: {WANTED_PATHS.stat().st_size/1e6:.2f} MB, "
      f"{len(sample):,} paths")
print(f"\nFirst three:")
print("\n".join(sample['archive_path'].head(3)))

sample_manifest.parquet: 2.27 MB
wanted_paths.txt: 3.60 MB, 75,000 paths

First three:
Project_CodeNet/data/p00000/C++/s582427538.cpp
Project_CodeNet/data/p00000/C++/s773824563.cpp
Project_CodeNet/data/p00000/C++/s110287235.cpp


In [9]:
chains = df.groupby(["user_id", "problem_id"], observed=True).agg(
    n=("submission_id", "size"),
    n_problem_split=("split", "nunique"),
    n_random_split=("random_split", "nunique"),
)
multi = chains["n"] > 1

print(f"(user, problem) chains:            {len(chains):,}")
print(f"  with more than one submission:   {multi.sum():,}")
print(f"  submissions inside those chains: {chains.loc[multi, 'n'].sum():,}")
print()
print(f"Chains straddling splits — problem-level: {(chains.loc[multi, 'n_problem_split'] > 1).sum():,}")
print(f"Chains straddling splits — random:        {(chains.loc[multi, 'n_random_split'] > 1).sum():,}")

(user, problem) chains:            3,992,111
  with more than one submission:   1,364,569
  submissions inside those chains: 5,012,514

Chains straddling splits — problem-level: 0
Chains straddling splits — random:        979,741
